# VoiceForge Auto-Train

Train an RVC v2 voice model on Google Colab with one click.

**Before starting:**
1. In VoiceForge, click **Package for Colab** to zip your vocal data
2. Upload the zip to Google Drive
3. Copy the file ID from the sharing link
4. Fill in the form below and click **Runtime → Run all**

## 1. Configure

In [ ]:
#@title Training Settings

GDRIVE_FILE_ID = "" #@param {type:"string"}
MODEL_NAME = "my_voice" #@param {type:"string"}

print(f"Model: {MODEL_NAME}")
print(f"Drive File ID: {GDRIVE_FILE_ID}")

In [ ]:
#@title Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")
print("Drive mounted")

## 2. Setup Environment

In [ ]:
#@title Install system dependencies
!apt-get -qq update
!apt-get -qq install -y libsndfile1-dev ffmpeg unzip wget curl
print("System deps installed")

In [ ]:
#@title Download training dataset from Google Drive
import os
import zipfile

dataset_zip = "/content/dataset.zip"
dataset_dir = "/content/dataset"

if GDRIVE_FILE_ID:
    !gdown --id {GDRIVE_FILE_ID} -O {dataset_zip} --fuzzy
    print(f"Downloaded: {dataset_zip}")
    os.makedirs(dataset_dir, exist_ok=True)
    with zipfile.ZipFile(dataset_zip, 'r') as zf:
        zf.extractall(dataset_dir)
    print(f"Extracted to: {dataset_dir}")
    !ls -la {dataset_dir}
else:
    from google.colab import files
    uploaded = files.upload()
    zip_name = list(uploaded.keys())[0]
    !unzip -o {zip_name} -d {dataset_dir}

In [ ]:
#@title Clone RVC WebUI
RVC_DIR = "/content/RVC-WebUI"
if not os.path.exists(RVC_DIR):
    !git clone https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI.git {RVC_DIR}
else:
    print("RVC already cloned")
%cd {RVC_DIR}
print(f"Working dir: {RVC_DIR}")

In [ ]:
#@title Install Python dependencies
!pip install -r requirements.txt -q
!pip install -q gdown
print("Python deps installed")

In [ ]:
#@title Download pretrained models
assets_dir = "/content/assets"
os.makedirs(f"{assets_dir}/hubert", exist_ok=True)
os.makedirs(f"{assets_dir}/rmvpe", exist_ok=True)
os.makedirs(f"{assets_dir}/pretrained", exist_ok=True)

if not os.path.exists(f"{assets_dir}/hubert/hubert_base.pt"):
    !wget -q https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/hubert_base.pt \
        -O {assets_dir}/hubert/hubert_base.pt

if not os.path.exists(f"{assets_dir}/rmvpe/rmvpe.pt"):
    !wget -q https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/rmvpe.pt \
        -O {assets_dir}/rmvpe/rmvpe.pt

if not os.path.exists(f"{assets_dir}/pretrained/f0G40k.pth"):
    !wget -q https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained_v2/f0G40k.pth \
        -O {assets_dir}/pretrained/f0G40k.pth
    !wget -q https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained_v2/f0D40k.pth \
        -O {assets_dir}/pretrained/f0D40k.pth

print("Pretrained models ready")

## 3. Preprocess Dataset

In [ ]:
#@title Preprocess training data
%%capture
!python trainset_preprocess_pitch_print.py \
    --data_dir {dataset_dir} \
    --preset 0 \
    --exp_dir /content/experiments/{MODEL_NAME}
print("Preprocessing complete")

## 4. Train Model (30-60 min on T4)

In [ ]:
#@title Start training
!python train.py \
    --exp_dir1 /content/experiments/{MODEL_NAME} \
    --pretrain_G /content/assets/pretrained/f0G40k.pth \
    --pretrain_D /content/assets/pretrained/f0D40k.pth
print("Training complete!")

## 5. Export Model to Google Drive

In [ ]:
#@title Save .pth and .index to your Drive
import shutil
import glob

drive_dir = f"/content/drive/MyDrive/VoiceForge/{MODEL_NAME}"
os.makedirs(drive_dir, exist_ok=True)

exp_dir = f"/content/experiments/{MODEL_NAME}"

pth_files = glob.glob(f"{exp_dir}/*.pth") + glob.glob(f"{exp_dir}/checkpoints/*.pth")
if pth_files:
    latest = max(pth_files, key=os.path.getmtime)
    shutil.copy(latest, f"{drive_dir}/{MODEL_NAME}.pth")
    print(f"Copied: {latest}")
else:
    print("No .pth found in expected paths, searching...")

index_files = glob.glob(f"{exp_dir}/*.index") + glob.glob(f"{exp_dir}/index/*.index")
for f in index_files:
    shutil.copy(f, f"{drive_dir}/{MODEL_NAME}.index")
    print(f"Copied index: {f}")

print(f"\nModel saved to: {drive_dir}")
print("Download .pth and .index files and drop into VoiceForge models folder.")

## Done!

1. Open **Google Drive** → `VoiceForge/{MODEL_NAME}`
2. Download `.pth` and `.index` files
3. Drop them into **VoiceForge → models** folder
4. Model appears automatically on Convert and TTS pages